# Три запроса: английский, русский, исправление черновика
Контроль на четырёх прослушанных примерах. 3/3 означает совпадение слов после нормализации регистра, пунктуации и ё/е, а не доказанную правильность. Прежние расхождения сохраняются. Автоматическое принятие отключено.


In [ ]:
import json, hashlib
import pandas as pd
from IPython.display import Audio, display
from lalm.annotation.inspection import open_snapshot
s3, snapshot, cache, db = open_snapshot()
run_prefix = 'debug/precision-triad-6f3319f29d81'
report = json.loads(s3.get_object(Bucket='speech-annotations', Key=run_prefix+'/report.json')['Body'].read())
pd.set_option('display.max_colwidth', 180)
rows = pd.DataFrame(report['results'])
display(rows[['sample_id','variant','language','transcript','error']])

display(pd.DataFrame(report['examples'])[['sample_id','three_texts_match','languages_agree','has_uncertainty','triad_candidate','prior_text_conflict','automatic_acceptance']])


In [ ]:
# Все четыре примера: замечание пользователя, ответы, аудио.
for example in report['examples']:
    print(example['sample_id'])
    print('Проверка пользователем:', example['feedback']['note'])
    print('Подтверждённый текст:', example['feedback']['verbatim_text'])
    print('Исходная разметка (может быть неверной):', example['original_annotation'])
    display(rows[rows.sample_id == example['sample_id']][['variant','language','transcript','error']])
    data = s3.get_object(Bucket=example['audio_bucket'],Key=example['audio_key'])['Body'].read()
    assert hashlib.sha256(data).hexdigest() == example['audio_sha256']
    display(Audio(data))


In [ ]:
# Точные инструкции всех вариантов; это только чтение, новые запросы не запускаются.
for row in report['results']:
    print(row['sample_id'][:12], row['variant'])
    request = json.loads(s3.get_object(Bucket='speech-annotations',Key=row['artifact_prefix']+'/request.json')['Body'].read())
    print(request['prompt'], '\n')
